# 5.1 Corriente Eléctrica

La corriente eléctrica se define fundamentalmente como un flujo o movimiento de cargas a través de un material. Si consideramos una sección transversal de área $A$, la corriente media $\bar{I}$ se define como la cantidad de carga $\Delta Q$ que atraviesa dicha área en un intervalo de tiempo $\Delta t$:

$$\bar{I} = \frac{\Delta Q}{\Delta t}$$

Dimensionalmente, la corriente es carga dividida por tiempo. En el Sistema Internacional, su unidad es el Amperio (A), donde $1\text{ A} = 1\text{ C/s}$. De forma instantánea, la corriente se expresa como:

$$I = \frac{dQ}{dt}$$

Es importante notar que un flujo de electrones moviéndose hacia la izquierda es equivalente a considerar protones moviéndose hacia la derecha bajo la presencia de un campo eléctrico $\vec{E}$.

## Modelo microscópico

Para entender la corriente a nivel microscópico, consideremos un volumen diferencial de un material conductor cilíndrico, dado por $dV = A \, dx$. Si la densidad volumétrica de portadores de carga es $n$ (número de cargas $N$ por unidad de volumen), la carga diferencial contenida en este volumen es:

$$\begin{aligned} Q &= N \, q \\
dQ &= q \, dN \\
dQ &= q \, n \, dV \\
dQ &= q \, n \, A \, dx \end{aligned}$$

Reemplazando esto en la definición de corriente:

$$I = \frac{dQ}{dt} = q \, n \, A \left(\frac{dx}{dt}\right)$$

La dinámica de una carga individual de masa $m$ dentro del conductor está gobernada por la fuerza eléctrica y una fuerza de "fricción" o amortiguamiento interna proporcional a la velocidad ($f = \gamma v$), generada por las colisiones con la red atómica. Usando la Segunda Ley de Newton ($\vec{F}_{\text{Neta}} = m\vec{a}$):

$$qE - \gamma v = m\frac{dv}{dt}$$

Reordenando e integrando desde el reposo ($t=0, v=0$):

$$\begin{aligned} \int_0^t dt &= m \int_0^v \frac{dv}{qE - \gamma v} \\
t &= -\frac{m}{\gamma} \ln\left(\frac{qE - \gamma v}{qE}\right)\end{aligned}$$

Despejando la velocidad $v(t)$:

$$v(t) = \frac{qE}{\gamma} \left(1 - e^{-\frac{\gamma}{m}t}\right)$$

Con el paso del tiempo, la exponencial decae y la carga alcanza una **velocidad de deriva** ($v_d$) constante:

$$v_d = \frac{qE}{\gamma}$$

> **Nota:** En metales, esta velocidad de deriva es sumamente baja, del orden de $10^{-3}$ a $10^{-4}\text{ m/s}$).


In [1]:
from IPython.display import display, HTML

simulacion_deriva_v3_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50;">Simulación Microscópica: Modelo de Drude</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px;">Ajusta el campo eléctrico, la densidad de la red y la velocidad térmica (temperatura) para observar su efecto en el flujo neto de electrones.</p>
    
    <div style="display: flex; gap: 20px; flex-wrap: wrap; margin-bottom: 15px;">
        <div style="flex: 1.5; min-width: 300px;">
            <div style="margin-bottom: 8px;">
                <label style="font-weight: 600; display: inline-block; width: 180px;">Campo Eléctrico (E): <span id="drude_valE_v3" style="font-weight: normal;">0</span></label>
                <input type="range" id="drude_E_v3" min="0" max="100" step="1" value="0" style="width: 40%; vertical-align: middle;">
            </div>
            <div style="margin-bottom: 8px;">
                <label style="font-weight: 600; display: inline-block; width: 180px;">Densidad de Red: <span id="drude_valIons_v3" style="font-weight: normal;">Media</span></label>
                <input type="range" id="drude_ions_v3" min="1" max="3" step="1" value="2" style="width: 40%; vertical-align: middle;">
            </div>
            <div style="margin-bottom: 8px;">
                <label style="font-weight: 600; display: inline-block; width: 180px;">Velocidad Térmica: <span id="drude_valTh_v3" style="font-weight: normal;">2.0</span></label>
                <input type="range" id="drude_th_v3" min="0.5" max="6.0" step="0.5" value="2.0" style="width: 40%; vertical-align: middle;">
            </div>
        </div>

        <div style="flex: 1; min-width: 220px; font-size: 0.95em; background: #fff; padding: 10px; border-radius: 6px; border: 1px solid #ccc;">
            <div style="margin-bottom: 5px;"><b>Velocidad de Deriva (vd):</b> <span id="drude_valVd_v3" style="color: #dc3545; font-weight: bold;">0.000</span> au</div>
            <div style="margin-bottom: 5px;"><b>Corriente Relativa (I):</b> <span id="drude_valC_v3" style="color: #0d6efd; font-weight: bold;">0.00</span> au</div>
            <div style="font-size: 0.8em; color: #777; margin-top: 5px;">*Velocidad térmica simula la T° del conductor</div>
        </div>
    </div>
    
    <div style="text-align: center; position: relative;">
        <canvas id="drude_canvas_v3" width="600" height="300" style="background: #eef2f5; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>
</div>

<script>
    setTimeout(function() {
        const canvas = document.getElementById('drude_canvas_v3');
        if (!canvas) return;
        const ctx = canvas.getContext('2d');
        
        const slider_E = document.getElementById('drude_E_v3');
        const slider_Ions = document.getElementById('drude_ions_v3');
        const slider_Th = document.getElementById('drude_th_v3');
        
        const val_E = document.getElementById('drude_valE_v3');
        const val_Ions = document.getElementById('drude_valIons_v3');
        const val_Th = document.getElementById('drude_valTh_v3');
        const val_Vd = document.getElementById('drude_valVd_v3');
        const val_C = document.getElementById('drude_valC_v3');
        
        let E_field = 0;
        let ion_density = 2; 
        let thermal_velocity = 2.0; 
        
        // Variable para el filtro de suavizado
        let smoothed_vd = 0;
        
        const num_electrons = 150;
        let electrons = [];
        let ions = [];
        
        function initIons() {
            ions = [];
            const rows = 3 * ion_density;
            const cols = 6 * ion_density;
            const dx = canvas.width / cols;
            const dy = canvas.height / rows;
            const radius = 8 + (4 - ion_density); 
            
            for(let i = 0; i <= cols; i++) {
                for(let j = 0; j <= rows; j++) {
                    let offsetX = (Math.random() - 0.5) * 10;
                    let offsetY = (Math.random() - 0.5) * 10;
                    ions.push({x: i*dx + offsetX, y: j*dy + offsetY, r: radius});
                }
            }
        }
        
        function initElectrons() {
            electrons = [];
            for(let i=0; i<num_electrons; i++) {
                let angle = Math.random() * 2 * Math.PI;
                electrons.push({
                    x: Math.random() * canvas.width,
                    y: Math.random() * canvas.height,
                    vx: Math.cos(angle) * thermal_velocity,
                    vy: Math.sin(angle) * thermal_velocity,
                    r: 2
                });
            }
        }

        function updateParams() {
            E_field = parseFloat(slider_E.value) / 1000; 
            let new_density = parseInt(slider_Ions.value);
            thermal_velocity = parseFloat(slider_Th.value);
            
            val_E.innerText = slider_E.value;
            val_Th.innerText = thermal_velocity.toFixed(1);
            
            let densityText = ["Baja", "Media", "Alta"];
            val_Ions.innerText = densityText[new_density - 1];
            
            if(new_density !== ion_density) {
                ion_density = new_density;
                initIons();
            }
        }

        function animate() {
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            
            ctx.fillStyle = '#dc3545';
            ions.forEach(ion => {
                ctx.beginPath();
                ctx.arc(ion.x, ion.y, ion.r, 0, Math.PI * 2);
                ctx.fill();
            });
            
            let total_vx = 0;

            ctx.fillStyle = '#0d6efd';
            electrons.forEach(e => {
                e.vx += E_field;
                
                e.x += e.vx;
                e.y += e.vy;
                
                if(e.x > canvas.width) e.x = 0;
                if(e.x < 0) e.x = canvas.width;
                if(e.y > canvas.height) { e.y = canvas.height; e.vy *= -1; }
                if(e.y < 0) { e.y = 0; e.vy *= -1; }
                
                ions.forEach(ion => {
                    let dx = e.x - ion.x;
                    let dy = e.y - ion.y;
                    let dist = Math.sqrt(dx*dx + dy*dy);
                    
                    if(dist < ion.r + e.r) {
                        let angle = Math.random() * 2 * Math.PI;
                        e.vx = Math.cos(angle) * thermal_velocity;
                        e.vy = Math.sin(angle) * thermal_velocity;
                        
                        e.x += (dx/dist) * 2;
                        e.y += (dy/dist) * 2;
                    }
                });
                
                total_vx += e.vx;
                
                ctx.beginPath();
                ctx.arc(e.x, e.y, e.r, 0, Math.PI * 2);
                ctx.fill();
            });
            
            // Cálculo instantáneo
            let instant_vd = total_vx / num_electrons;
            if(slider_E.value == 0) instant_vd = 0; 
            
            // Aplicación del filtro de media móvil (suavizado exponencial)
            // El factor 0.05 define qué tan suave es (valores menores = más estable pero más lento en reaccionar)
            smoothed_vd = (0.05 * instant_vd) + (0.95 * smoothed_vd);
            
            // Forzar a 0 absoluto si no hay campo, para que no quede un remanente infinitesimal
            if(slider_E.value == 0 && Math.abs(smoothed_vd) < 0.01) smoothed_vd = 0;
            
            val_Vd.innerText = Math.abs(smoothed_vd).toFixed(3);
            val_C.innerText = (Math.abs(smoothed_vd) * num_electrons * 0.1).toFixed(2); 

            requestAnimationFrame(animate);
        }

        slider_E.addEventListener('input', updateParams);
        slider_Ions.addEventListener('input', updateParams);
        slider_Th.addEventListener('input', updateParams);

        initIons();
        initElectrons();
        updateParams();
        animate();
    }, 150); 
</script>
"""
display(HTML(simulacion_deriva_v3_html))


## Preguntas de Análisis


In [2]:
from IPython.display import display, HTML

quiz_corriente_html = """
<style>
    .fisi-quiz-container {
        border: 1px solid #e0e0e0;
        padding: 15px;
        margin-bottom: 25px;
        border-radius: 8px;
        background-color: #f8f9fa;
        font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif;
    }
    .fisi-question {
        font-weight: 600;
        margin-bottom: 15px;
        color: #2c3e50;
    }
    .fisi-option {
        margin-bottom: 10px;
        display: flex;
        align-items: flex-start;
        cursor: pointer;
    }
    .fisi-option input {
        margin-top: 4px;
        margin-right: 10px;
    }
    .fisi-check-btn {
        background-color: #f73bc5;
        color: white;
        border: none;
        padding: 8px 20px;
        border-radius: 4px;
        cursor: pointer;
        font-size: 14px;
        margin-top: 10px;
        font-weight: 500;
    }
    .fisi-check-btn:hover { background-color: #ab2988; }
    .fisi-feedback {
        margin-top: 15px;
        padding: 12px;
        border-radius: 6px;
        display: none;
        font-size: 0.95em;
    }
</style>

<script>
    if (typeof window.checkFisiQuiz === 'undefined') {
        window.checkFisiQuiz = function(radioName, feedbackId) {
            var radios = document.getElementsByName(radioName);
            var selected = null;
            var feedbackDiv = document.getElementById(feedbackId);
            
            for (var i = 0; i < radios.length; i++) {
                if (radios[i].checked) {
                    selected = radios[i];
                    break;
                }
            }
            
            feedbackDiv.style.display = 'block';
            
            if (!selected) {
                feedbackDiv.style.backgroundColor = '#fff3cd';
                feedbackDiv.style.color = '#856404';
                feedbackDiv.style.border = '1px solid #ffeeba';
                feedbackDiv.innerHTML = '⚠️ Por favor, selecciona una opción.';
                return;
            }
            
            var isCorrect = selected.getAttribute('data-correct') === 'true';
            var msg = selected.getAttribute('data-fb');
            
            if (isCorrect) {
                feedbackDiv.style.backgroundColor = '#d4edda';
                feedbackDiv.style.color = '#155724';
                feedbackDiv.style.border = '1px solid #c3e6cb';
                feedbackDiv.innerHTML = '✅ <strong>¡Correcto!</strong><br>' + msg;
            } else {
                feedbackDiv.style.backgroundColor = '#f8d7da';
                feedbackDiv.style.color = '#721c24';
                feedbackDiv.style.border = '1px solid #f5c6cb';
                feedbackDiv.innerHTML = '❌ <strong>Incorrecto.</strong><br>' + msg;
            }
        };
    }
</script>

<div class="fisi-quiz-container">
    <div class="fisi-question">
        1. Si en un alambre metálico observamos un flujo neto de electrones (cargas negativas) moviéndose hacia la izquierda debido a la presencia de un campo eléctrico, ¿cómo se define convencionalmente el sentido de la corriente eléctrica (I)?
    </div>
    <label class="fisi-option">
        <input type="radio" name="qcorr1" value="a" data-fb="Incorrecto. Aunque el flujo físico real sea de electrones hacia la izquierda, la convención histórica dicta lo contrario."> 
        <span>Hacia la izquierda, ya que el vector de corriente eléctrica siempre debe apuntar en la dirección real del flujo de las cargas libres mayoritarias.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qcorr1" value="b" data-correct="true" data-fb="¡Exacto! Por convención histórica (establecida antes de conocerse los electrones), la dirección de la corriente eléctrica se define como el sentido en el que fluirían los portadores de carga positivos. Por ende, un flujo de electrones hacia la izquierda es matemáticamente equivalente a una corriente positiva hacia la derecha."> 
        <span>Hacia la derecha, porque por convención se define la corriente en la dirección en la que se moverían los portadores de carga positivos bajo la influencia de ese mismo campo eléctrico.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qcorr1" value="c" data-fb="Incorrecto. Los electrones en movimiento constituyen una corriente real y medible. No es necesario que haya protones moviéndose para que exista corriente eléctrica."> 
        <span>Es nula, puesto que los electrones tienen carga negativa y solo un movimiento físico de partículas positivas puede generar una corriente eléctrica medible.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qcorr1" value="d" data-fb="Incorrecto. Si el campo eléctrico es unidireccional y constante, el movimiento neto de cargas producirá una corriente continua (DC), no alterna."> 
        <span>Alternante, ya que la corriente cambia de sentido periódicamente en cualquier conductor que posea una red atómica rígida.</span>
    </label>
    <button class="fisi-check-btn" onclick="checkFisiQuiz('qcorr1', 'feedback-corr1')">Comprobar</button>
    <div class="fisi-feedback" id="feedback-corr1"></div>
</div>

<div class="fisi-quiz-container">
    <div class="fisi-question">
        2. De acuerdo con la deducción microscópica (I = q·n·A·v_d), si logramos sustituir el material del conductor por uno distinto que posea el doble de densidad volumétrica de portadores de carga libres (n), asumiendo que el área transversal (A) y la velocidad de deriva (v_d) logran mantenerse iguales, ¿qué ocurrirá con la corriente total I?
    </div>
    <label class="fisi-option">
        <input type="radio" name="qcorr2" value="a" data-correct="true" data-fb="¡Correcto! En la ecuación I = q·n·A·v_d, la corriente I es directamente proporcional a 'n' (la cantidad de cargas libres disponibles por unidad de volumen). Al haber el doble de portadores moviéndose a la misma velocidad por la misma área, la cantidad de carga que atraviesa la sección por segundo se duplica."> 
        <span>Se duplicará, ya que la corriente es directamente proporcional a la densidad volumétrica de portadores (n).</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qcorr2" value="b" data-fb="Incorrecto. Una mayor densidad de portadores libres ('n') no significa necesariamente un aumento en la fricción (γ). El enunciado asume explícitamente que la velocidad de deriva v_d se logra mantener."> 
        <span>Se reducirá a la mitad, porque un material más denso aumenta las colisiones internas, dificultando severamente el paso de la corriente.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qcorr2" value="c" data-fb="Incorrecto. La velocidad de deriva v_d indica qué tan rápido se mueve el 'enjambre', pero 'n' indica cuántas partículas conforman ese enjambre. Ambas variables impactan la corriente total."> 
        <span>Permanecerá constante, debido a que la velocidad de deriva (v_d) determina de manera exclusiva la magnitud final de la corriente en un circuito.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qcorr2" value="d" data-fb="Incorrecto. La relación entre la corriente (I) y la densidad de portadores (n) es estrictamente lineal, no cuadrática."> 
        <span>Se cuadruplicará, ya que al aumentar la densidad (n) también aumenta geométricamente la probabilidad de colisión entre cargas libres.</span>
    </label>
    <button class="fisi-check-btn" onclick="checkFisiQuiz('qcorr2', 'feedback-corr2')">Comprobar</button>
    <div class="fisi-feedback" id="feedback-corr2"></div>
</div>

<div class="fisi-quiz-container">
    <div class="fisi-question">
        3. En el modelo microscópico descrito, la velocidad de deriva (v_d) que alcanzan los electrones en un cable de cobre doméstico suele ser diminuta (del orden de milímetros por segundo). Sin embargo, al encender un interruptor, la ampolleta de una habitación se ilumina de forma prácticamente instantánea. ¿Cómo explica la física esta aparente contradicción?
    </div>
    <label class="fisi-option">
        <input type="radio" name="qcorr3" value="a" data-fb="Incorrecto. Los electrones tienen masa y no pueden alcanzar la velocidad de la luz en el vacío (c). El valor v_d = milímetros por segundo es real, no una ficción matemática."> 
        <span>Porque los electrones individuales en realidad viajan a la velocidad de la luz, y la ecuación de Drude solo es un promedio estadístico ficticio que no representa el movimiento real.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qcorr3" value="b" data-correct="true" data-fb="¡Exacto! Aunque el movimiento neto de cada electrón (velocidad de deriva) es muy lento, al cerrar el circuito se establece un campo eléctrico que se propaga por todo el alambre a una velocidad cercana a la de la luz. Este campo empuja casi simultáneamente a TODOS los electrones libres que ya estaban dentro del filamento de la ampolleta, generando corriente instantáneamente sin necesidad de esperar a que un electrón viaje desde el interruptor."> 
        <span>Porque lo que viaja casi a la velocidad de la luz por el cable es el campo eléctrico, el cual pone en movimiento simultáneo a todos los electrones del circuito, incluyendo a los que ya estaban en la ampolleta.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qcorr3" value="c" data-fb="Incorrecto. La fricción con la red cristalina (colisiones) no desaparece al inicio; está presente en todo momento y es lo que evita que los electrones se aceleren infinitamente."> 
        <span>Porque la fuerza de fricción atómica desaparece en la primera fracción de segundo del encendido, permitiendo que la nube de electrones adquiera una aceleración momentánea infinita.</span>
    </label>
    <label class="fisi-option">
        <input type="radio" name="qcorr3" value="d" data-fb="Incorrecto. En los conductores sólidos metálicos, los núcleos atómicos (protones) están rígidamente unidos en la red cristalina y no contribuyen en absoluto a la conducción de la corriente."> 
        <span>Porque mientras los electrones se mueven lentamente, los protones del metal se desplazan velozmente en la dirección opuesta, siendo ellos los que transportan la energía inicial.</span>
    </label>
    <button class="fisi-check-btn" onclick="checkFisiQuiz('qcorr3', 'feedback-corr3')">Comprobar</button>
    <div class="fisi-feedback" id="feedback-corr3"></div>
</div>
"""

display(HTML(quiz_corriente_html))